# ALS applications

## Dzen dataset

Data comes from [dzen.ru](https://dzen.ru/) site and consists of likes which users put to text articles

### Columns
1. item_id - unique id of an item (article)
2. user_id - unique id of a user
3. source_id - unique id of an author. If two items have same source_id, then they come from one author
4. Name of item is name of the article
5. Raw dataset represents user_id and list of item_ids which user liked

In [1]:
!curl -O -J -L 'https://www.dropbox.com/s/ia4bvhuqg8kesee/zen_dataset.zip?dl=1'
!unzip unspecified

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   109    0   109    0     0    452      0 --:--:-- --:--:-- --:--:--   454
100   261    0   261    0     0    395      0 --:--:-- --:--:-- --:--:--   395
100   496    0   496    0     0    441      0 --:--:--  0:00:01 --:--:--  1158
100 24.0M  100 24.0M    0     0  8637k      0  0:00:02  0:00:02 --:--:-- 27.0M
Archive:  unspecified
  inflating: zen_item_to_name.csv    
  inflating: zen_item_to_source.csv  
  inflating: zen_ratings.csv         


In [2]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm.notebook import tqdm
from scipy.sparse.linalg import inv
from scipy.sparse import eye, diags
from scipy.sparse.linalg import spsolve
from numpy.linalg import solve
import ast

In [3]:
item_names = pd.read_csv("zen_item_to_name.csv")
item_sources = pd.read_csv("zen_item_to_source.csv")
dataset = pd.read_csv("zen_ratings.csv", converters={'item_ids': ast.literal_eval})

In [4]:
item_names

,id,name
0,94962,Что обычно ожидало русских казачек в руках у к...
1,3972,Почему Россия решила строить новую скоростную ...
2,94644,"5 неприличных фактов об Андрее Макаревиче, кот..."
3,82518,"Что стало с красавицей Хмельницкой, которую му..."
4,53264,"Понять и Простить: Почему угонщики, бежавшие и..."
...,...,...
104498,36769,"Плюс один источник мифа о рыцарях, неспособных..."
104499,9190,Мой сад - малоуходный
104500,52731,Купил первую в жизни циркулярную пилу. Честный...
104501,72660,Решили предложить Марине помощь в лечении ч.10


In [5]:
item_sources

,id,source
0,94962,2919814402697966089
1,3972,3263022753228392991
2,94644,-3857390427602554682
3,82518,-9036908390349249792
4,53264,3353856219169766284
...,...,...
104498,36769,3818746211375738614
104499,9190,4975535765688979937
104500,52731,3720366796439288909
104501,72660,-7860042973720636310


In [6]:
dataset

,user_id,item_ids
0,993675863667353526,"[15267, 61075, 81203, 17066, 25471, 88427, 638..."
1,4250619547882954185,"[4555, 94644, 84972, 17774, 94962, 78217, 2485..."
2,3847785305345691076,"[1898, 26703, 16525, 86939, 55017, 31069, 4035..."
3,1785181112918558233,"[75601, 102458, 28716, 100694, 5757, 47104, 60..."
4,5078748097863903181,"[72260, 40825, 2615, 42549, 379, 100818, 56827..."
...,...,...
75905,4954138831959898373,"[11881, 55520, 63054, 48015, 66952, 103830, 21..."
75906,4967793435819938014,"[74697, 11830, 63858, 87245, 41956, 62089, 686..."
75907,7137764184903122777,"[10353, 1775, 103680, 29704, 9782, 13295, 9975..."
75908,2624987805086334956,"[24324, 18854, 73319, 66641, 64078, 97387, 426..."


In [7]:
total_interactions_count = dataset.item_ids.map(len).sum()
user_coo = np.zeros(total_interactions_count, dtype=np.int64)
item_coo = np.zeros(total_interactions_count, dtype=np.int64)
pos = 0

for user_id, item_ids in enumerate(tqdm(dataset.item_ids)):
    user_coo[pos : pos + len(item_ids)] = user_id
    item_coo[pos : pos + len(item_ids)] = item_ids
    pos += len(item_ids)

shape = (max(user_coo) + 1, max(item_coo) + 1)
user_item_matrix = sp.coo_matrix(
    (np.ones(len(user_coo)), (user_coo, item_coo)), shape=shape
)
user_item_matrix = user_item_matrix.tocsr()
sp.save_npz("data_train.npz", user_item_matrix)
# Cleanup memory. Later you need just data_train.npz
del user_coo
del item_coo
del dataset

  0%|          | 0/75910 [00:00<?, ?it/s]

In [8]:
# you could start here if you already done precomputing
user_item_matrix = sp.load_npz("data_train.npz")

In [9]:
user_item_matrix

<75910x104503 sparse matrix of type '<class 'numpy.float64'>'
	with 5792423 stored elements in Compressed Sparse Row format>

In [10]:
def sparce_matrix_report(matrix):
    print('Size of raw data:', matrix.data.nbytes / 10**6, 'Mb')
    print('Feedback matrix size:', matrix.shape)

In [11]:
sparce_matrix_report(user_item_matrix)

Size of raw data: 46.339384 Mb
Feedback matrix size: (75910, 104503)


In [12]:
item_weights = np.array(user_item_matrix.tocsc().sum(0))[0]
top_to_bottom_order = np.argsort(-item_weights)
item_mapping = np.empty(top_to_bottom_order.shape, dtype=int)
item_mapping[top_to_bottom_order] = np.arange(len(top_to_bottom_order))
total_item_count = (item_weights > 0).sum()
total_user_count = user_item_matrix.shape[0]


def build_debug_dataset(user_item_matrix, item_pct: float, user_pct: float):
    '''Get given percent of top rated items and given percent of random users'''
    user_count = int(total_user_count * user_pct),
    item_count = int(total_item_count * item_pct)
    item_ids = top_to_bottom_order[:item_count]
    user_ids = np.random.choice(
        np.arange(user_item_matrix.shape[0]), size=user_count, replace=False
    )
    train = user_item_matrix[user_ids]
    train = train[:, item_ids]
    return train

In [13]:
debug_dataset = build_debug_dataset(user_item_matrix, 0.05, 0.05)

sparce_matrix_report(debug_dataset)

Size of raw data: 1.078792 Mb
Feedback matrix size: (3795, 5019)


This is useful for debugging (just to save time).

**Final answers should use full dataset!!!**

## Split dataset matrix (5 points)

in the following way: for 20% of users (random) remove one like - this will be test data. The rest is train data. (10 points)

In [14]:
def split_data(ratings):
    # Ensure ratings is in a mutable format
    train_matrix = ratings.copy()
    # Create an empty matrix with the same shape as ratings for the test set
    test_matrix = sp.lil_matrix(ratings.shape)

    # Number of users in the matrix
    num_users = ratings.shape[0]

    # Select 20% of the users to be in the test set
    num_test_users = int(num_users * 0.2)
    test_user_indices = np.random.choice(num_users, size=num_test_users, replace=False)

    # Iterate over each test user and remove one like
    for user in test_user_indices:
        # Get the indices of items that this user has liked
        item_indices = train_matrix[user].nonzero()[1]

        if len(item_indices) > 0:
            # Randomly select one item index to remove
            item_to_remove = np.random.choice(item_indices)
            # Set the training interaction to 0
            train_matrix[user, item_to_remove] = 0
            # Set the test interaction to 1
            test_matrix[user, item_to_remove] = 1

    # Convert matrices to CSR format after modification
    train_matrix = train_matrix.tocsr()
    test_matrix = test_matrix.tocsr()

    return train_matrix, test_matrix

In [15]:
train_ratings, test_ratings = split_data(user_item_matrix)


## Implement ALS, IALS (10 points each)

Note that due to size of data you need to implement algorithms with _sparce matrices_!

In [22]:
def als(ratings, k: int, lam: float):
    '''Alternating Least Squares algorithm
    Args:
        ratings: sparse matrix of ratings (assumed to be in CSR format for efficient row operations)
        k: size of embeddings
        lam: regularization term
    Returns:
        two matrices: of user embeddings and of item embeddings
    '''
    n_users, n_items = ratings.shape
    user_embeddings = np.random.rand(n_users, k)
    item_embeddings = np.random.rand(n_items, k)
    I = lam * np.eye(k)

    for iteration in range(10):  # Assuming some fixed number of iterations
        for u in range(n_users):
            # Compute this user's item ratings vector
            rated_items = ratings[u].nonzero()[1]
            R_u = ratings[u, rated_items].toarray()

            # Get the current item factors for these items
            F_i = item_embeddings[rated_items, :]

            # Solve for this user's embedding
            A_u = F_i.T @ F_i + I
            V_u = F_i.T @ R_u.T
            user_embeddings[u] = np.linalg.solve(A_u, V_u).flatten()  # Make sure the shapes align and are flat

        for i in range(n_items):
            # Compute this item's user ratings vector
            interested_users = ratings[:, i].nonzero()[0]
            R_i = ratings[interested_users, i].toarray()

            # Get the current user factors for these users
            F_u = user_embeddings[interested_users, :]

            # Solve for this item's embedding
            A_i = F_u.T @ F_u + I
            V_i = F_u.T @ R_i
            item_embeddings[i] = np.linalg.solve(A_i, V_i).flatten()  # Flatten as solve returns a 2D array for a single column

    return user_embeddings, item_embeddings


In [20]:
def ials(ratings, k: int, lam: float):
    '''Implicit Alternating Least Squares algorithm

    Args:
        ratings: sparse matrix of ratings (CSR format)
        k: size of embeddings
        lam: regularization term

    Returns:
        two matrices: of user embeddings and of item embeddings
    '''
    num_users, num_items = ratings.shape
    user_embeddings = np.random.rand(num_users, k)
    item_embeddings = np.random.rand(num_items, k)
    alpha = 40
    confidence = ratings.copy()
    confidence.data = 1 + alpha * confidence.data  # Only update non-zero entries

    lambda_eye = lam * np.eye(k)

    for iteration in range(10):  # Fixed number of iterations
        for u in range(num_users):
            C_u = diags(confidence[u].toarray()[0], 0)
            A_u = item_embeddings.T @ C_u @ item_embeddings + lambda_eye
            b_u = item_embeddings.T @ C_u @ np.ones(num_items)
            user_embeddings[u] = solve(A_u, b_u)  # A_u and b_u are dense here

        for i in range(num_items):
            C_i = diags(confidence[:, i].toarray()[0], 0)
            A_i = user_embeddings.T @ C_i @ user_embeddings + lambda_eye
            b_i = user_embeddings.T @ C_i @ np.ones(num_users)
            item_embeddings[i] = solve(A_i, b_i)  # A_i and b_i are dense here

    return user_embeddings, item_embeddings


## Compute MRR@100 metric for test users (10 points)

For ALS and IALS algorithms.

**Don't forget to use full dataset!**

In [18]:
#To get predictions, als_predictions, and ials_predictions as asked below, there is need to program it
def compute_predictions(user_embeddings, item_embeddings):
    # Compute the matrix multiplication of user embeddings and item embeddings transpose
    predictions = user_embeddings @ item_embeddings.T
    return predictions


In [ ]:
#Takes time to run
user_embeddings_als, item_embeddings_als = als(user_item_matrix, k=10, lam=0.1)

In [ ]:
#Takes time to run
user_embeddings_ials, item_embeddings_ials = ials(user_item_matrix, k=10, lam=0.1)

In [ ]:
#runtime depends on the outcome of the above last two codes
als_predictions = compute_predictions(user_embeddings_als, item_embeddings_als)

ials_predictions = compute_predictions(user_embeddings_ials, item_embeddings_ials)


In [ ]:
def mrr(predictions, test_ratings):
    total_reciprocal_rank = 0
    num_users = predictions.shape[0]

    # For each user in the test dataset
    for user_index in range(num_users):
        # Get the predicted and actual ratings
        user_predictions = predictions[user_index]
        user_actual = test_ratings[user_index].indices

        # Calculate the top 100 predicted items (assuming predictions are scores and not already sorted indices)
        top_n_indices = np.argsort(user_predictions)[::-1][:100]

        # Find the rank of the first relevant item
        rank = 101  # Default rank if no relevant item is found in top 100
        for idx, item_idx in enumerate(top_n_indices, 1):
            if item_idx in user_actual:
                rank = idx
                break

        # Calculate reciprocal rank
        if rank <= 100:
            total_reciprocal_rank += 1 / rank

    # Compute the mean reciprocal rank
    mrr_value = total_reciprocal_rank / num_users
    return mrr_value


In [ ]:
mrr_als = mrr(als_predictions, test_ratings)
print(mrr_als)

In [ ]:
mrr_ials = mrr(ials_predictions, test_ratings)
print(mrr_ials)

## Adjust hyperparameters of ALS and IALS to maximize MRR (20 points)

Main hyperparameters are regularization and weights for implicit case.

In [ ]:
def grid_search_als(user_item_matrix, test_ratings, ks, lams):
    best_mrr = 0
    best_params = (None, None)
    for k in ks:
        for lam in lams:
            print(f"Testing ALS with k={k}, lambda={lam}")
            user_embeddings, item_embeddings = als(user_item_matrix, k, lam)
            als_predictions = compute_predictions(user_embeddings, item_embeddings)
            current_mrr = mrr(als_predictions, test_ratings)
            if current_mrr > best_mrr:
                best_mrr = current_mrr
                best_params = (k, lam)
                print(f"New best MRR: {best_mrr} with k={k}, lambda={lam}")
    return best_params, best_mrr

def grid_search_ials(user_item_matrix, test_ratings, ks, lams, alphas):
    best_mrr = 0
    best_params = (None, None, None)
    for k in ks:
        for lam in lams:
            for alpha in alphas:
                print(f"Testing IALS with k={k}, lambda={lam}, alpha={alpha}")
                user_embeddings, item_embeddings = ials(user_item_matrix, k, lam, alpha)
                ials_predictions = compute_predictions(user_embeddings, item_embeddings)
                current_mrr = mrr(ials_predictions, test_ratings)
                if current_mrr > best_mrr:
                    best_mrr = current_mrr
                    best_params = (k, lam, alpha)
                    print(f"New best MRR: {best_mrr} with k={k}, lambda={lam}, alpha={alpha}")
    return best_params, best_mrr

# Define ranges for hyperparameters
ks = [10, 20, 50]  # Possible dimensions for embeddings
lams = [0.01, 0.1, 1]  # Regularization strengths
alphas = [10, 40, 70]  # Confidence levels for IALS

# Perform grid search
best_als_params, best_als_mrr = grid_search_als(user_item_matrix, test_ratings, ks, lams)
print(f"Best ALS params: {best_als_params}, MRR: {best_als_mrr}")

best_ials_params, best_ials_mrr = grid_search_ials(user_item_matrix, test_ratings, ks, lams, alphas)
print(f"Best IALS params: {best_ials_params}, MRR: {best_ials_mrr}")


Optimal parameters of ALS are:

....

Optimal parameters of IALS are:

....

## Get similarities from item2item CF and SLIM (10 points each)

Item2item can be taken from the first homework, SLIM was implemented in the class.

Alternatively you could use libraries, but in this case you will need to convert dataset to their format.

You need to compute only item similarities, not predictions for users.

In [ ]:
i2i_similarities = ... # your code here

In [ ]:
slim_similarities = ... # your code here

## Compare similarities from four algorithms (20 points)

* plot distributions
* compute metrics (which you think are relevant)
* look at several top similar lists

Make conclusion how these methods differ in computing similarities

In [ ]:
# your code here

Conclusion:

....